# PCB Defect — Cross-Dataset Model Adaptation (DeepPCB → PCB-Defect v1)**Pipeline**| Stage | What it does ||---|---|| 0 | Train / locate the **source** YOLO26n on DeepPCB (6 classes) || 1 | Convert PCB-Defect v1 COCO → YOLO, stratified 70/15/15 split, build 10% / 20% train subsets || 2 | **Zero-shot** evaluation of the source model on the target test split (imgsz 640 & 1280) || 3 | **Fine-tune** at 10% / 20% / 100% of the target train split, 3 seeds each || 4 | Results tables + adaptation curve || 5 | **XAI** — EigenCAM, Grad-CAM, Grad-CAM++, HiResCAM + quantitative localization metrics |**Class handling (important)**Five defect families are shared between the datasets. The sixth class differs in *kind*:DeepPCB has `pin-hole` (a void inside copper), PCB-Defect v1 has `missing_pad` (an absent contact area).These are **not** synonyms and are not mapped onto each other.```idx  DeepPCB (source)   PCB-Defect v1 (target)   status 0   copper          →  spurious_copper          shared 1   mousebite       →  mouse_bite               shared 2   open            →  open_circuit             shared 3   pin-hole        ✗  missing_pad              dataset-specific SLOT 4   short           →  short                    shared 5   spur            →  spur                     shared```Index 3 is a *slot* holding each dataset's own sixth class. This keeps `nc=6` on both sides sofine-tuning reuses the full classification head.* **Zero-shot headline metric = 5-class mAP** over the shared families only. Class 3 is excluded,  because the source neuron means `pin-hole` and the target ground truth means `missing_pad`.* A **separate diagnostic** measures whether the `pin-hole` neuron fires on `missing_pad` regions.* **Fine-tuned metrics** are reported over all 6 classes *and* over the 5 shared classes, so the  zero-shot and fine-tuned columns stay directly comparable.> Run cells top to bottom. Enable GPU (Settings → Accelerator → GPU T4 x2 / P100).

## 0 · Environment

In [ ]:
import os, sys, subprocess

# Ultralytics 8.4.x is required for YOLO26.
try:
    import ultralytics
    print("ultralytics", ultralytics.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics
    print("installed ultralytics", ultralytics.__version__)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Keep Ultralytics writing everything inside /kaggle/working (input dirs are read-only).
from ultralytics import settings as ul_settings
ul_settings.update({"datasets_dir": "/kaggle/working/data", "runs_dir": "/kaggle/working/runs"})
os.environ["WANDB_DISABLED"] = "true"

## 1 · Configuration

In [ ]:
from pathlib import Path

# ---- input paths -----------------------------------------------------------
SRC_ROOT   = Path("/kaggle/input/datasets/skm566/deep-pcb-dataset-v1/DeepPCB-1")
TGT_COCO   = Path("/kaggle/input/datasets/skm566/pcb-defect-dataset/PCB_Defect/annotation/_annotations.coco.json")
TGT_IMGDIR = Path("/kaggle/input/datasets/skm566/pcb-defect-dataset/PCB_Defect/images")

# COCO-pretrained YOLO26n used as the starting point for source training
BASE_WEIGHTS = Path("/kaggle/input/models/skmuktadir/yolo-26-modified-trained-on-deeppcb/tensorflow2/default/3/yolo26n.pt")

# If you already have a DeepPCB-trained checkpoint (nc=6), point here to skip Stage 0 training.
PRETRAINED_SOURCE = None      # e.g. Path("/kaggle/input/.../best.pt")

# ---- output paths ----------------------------------------------------------
WORK    = Path("/kaggle/working")
DATA    = WORK / "data"
RUNS    = WORK / "runs"
OUT     = WORK / "outputs"
for d in (DATA, RUNS, OUT, OUT / "xai", OUT / "figs"):
    d.mkdir(parents=True, exist_ok=True)

# ---- class slots -----------------------------------------------------------
SOURCE_NAMES = ["copper", "mousebite", "open", "pin-hole", "short", "spur"]
TARGET_NAMES = ["spurious_copper", "mouse_bite", "open_circuit", "missing_pad", "short", "spur"]
SHARED_IDX   = [0, 1, 2, 4, 5]     # families present in BOTH datasets
SLOT_IDX     = 3                   # dataset-specific sixth class

# COCO category name -> target index (slot order above)
COCO_NAME_TO_IDX = {
    "spurious_copper": 0,
    "mouse_bite":      1,
    "open_circuit":    2,
    "missing_pad":     3,
    "short":           4,
    "short_circuit":   4,   # tolerate either spelling
    "spur":            5,
}

# ---- experiment settings ---------------------------------------------------
SPLIT_FRACS   = {"train": 0.70, "val": 0.15, "test": 0.15}
SPLIT_SEED    = 42
FRACTIONS     = [0.10, 0.20]       # add 1.00 for the full-data ceiling run
SEEDS         = [0]                # pilot pass; use [0, 1, 2] for mean +/- std
MIN_PER_CLASS = 5                  # floor when sampling a subset

# --- PILOT SETTINGS ---------------------------------------------------------
# Source training is a short 10-epoch pass. See the warning in Stage 0c: this
# produces a weak source model and therefore a pessimistic zero-shot number.
SRC_EPOCHS, SRC_BATCH, SRC_IMGSZ = 10, 32, 640
FT_EPOCHS,  FT_BATCH,  FT_IMGSZ  = 50, 8,  640
FT_PATIENCE = 10
ZS_IMGSZ    = [640, 1280]          # zero-shot evaluated at two input scales

print("config ok")

## 2 · Helpers`link_tree` mirrors a read-only dataset into `/kaggle/working` with symlinks so Ultralytics canwrite its `.cache` files. `per_class_table` / `subset_map` pull per-class metrics out of anUltralytics validation result and average them over a chosen class subset.

In [ ]:
import json, random, shutil, math
from collections import Counter, defaultdict
import numpy as np

def link_tree(src: Path, dst: Path):
    # mirror directory structure with symlinked files (fast, no disk copy)
    dst.mkdir(parents=True, exist_ok=True)
    for p in src.rglob("*"):
        rel = p.relative_to(src)
        q = dst / rel
        if p.is_dir():
            q.mkdir(parents=True, exist_ok=True)
        else:
            q.parent.mkdir(parents=True, exist_ok=True)
            if not q.exists():
                try:
                    q.symlink_to(p)
                except OSError:
                    shutil.copy2(p, q)
    return dst


def write_yaml(path: Path, train, val, test, names):
    lines = [f"train: {train}", f"val: {val}"]
    if test is not None:
        lines.append(f"test: {test}")
    lines.append(f"nc: {len(names)}")
    lines.append("names:")
    for i, n in enumerate(names):
        lines.append(f"  {i}: {n}")
    path.write_text("\n".join(lines) + "\n")
    return path


def per_class_table(metrics, names):
    # -> {class_idx: dict(P, R, mAP50, mAP50_95)}  for classes present in the eval
    out = {}
    idx = list(metrics.box.ap_class_index)
    for j, c in enumerate(idx):
        p, r, ap50, ap = metrics.box.class_result(j)
        out[int(c)] = {"name": names[int(c)], "P": float(p), "R": float(r),
                       "mAP50": float(ap50), "mAP50_95": float(ap)}
    return out


def subset_map(table, idxs):
    # mean of per-class AP over the given class indices (only those actually evaluated)
    rows = [table[i] for i in idxs if i in table]
    if not rows:
        return {"mAP50": float("nan"), "mAP50_95": float("nan"),
                "P": float("nan"), "R": float("nan"), "n_classes": 0}
    return {"mAP50":    float(np.mean([r["mAP50"] for r in rows])),
            "mAP50_95": float(np.mean([r["mAP50_95"] for r in rows])),
            "P":        float(np.mean([r["P"] for r in rows])),
            "R":        float(np.mean([r["R"] for r in rows])),
            "n_classes": len(rows)}


def show_table(table, title=""):
    if title:
        print(title)
    print(f"  {'class':<18}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>11}")
    for i in sorted(table):
        r = table[i]
        print(f"  {r['name']:<18}{r['P']:>8.3f}{r['R']:>8.3f}{r['mAP50']:>9.3f}{r['mAP50_95']:>11.3f}")

print("helpers ok")

## 3 · Stage 0a — locate or verify the source checkpointA DeepPCB-trained model reports `nc=6` with the defect names. A stock COCO checkpoint reports`nc=80`. This cell scans every `.pt` it can see and tells you which you have.

In [ ]:
import glob, torch

def inspect_ckpt(p):
    try:
        ck = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        return {"path": str(p), "error": str(e)[:120]}
    m = ck.get("model")
    ta = ck.get("train_args", {}) or {}
    return {"path": str(p),
            "nc": getattr(m, "nc", None),
            "names": list(getattr(m, "names", {}).values())[:8],
            "data": ta.get("data"),
            "epochs": ta.get("epochs"),
            "date": ck.get("date")}

cands = sorted(glob.glob("/kaggle/input/models/**/*.pt", recursive=True))
if PRETRAINED_SOURCE:
    cands.append(str(PRETRAINED_SOURCE))

source_ready = None
for c in cands:
    info = inspect_ckpt(c)
    print(json.dumps(info, default=str, indent=2))
    if info.get("nc") == 6:
        source_ready = Path(c)
    print("-" * 70)

if source_ready:
    print(f"FOUND a 6-class checkpoint -> {source_ready}")
    print("Stage 0c (DeepPCB training) will be SKIPPED.")
else:
    print("No 6-class checkpoint found. Stage 0c will train YOLO26n on DeepPCB.")
    assert BASE_WEIGHTS.exists(), f"missing base weights: {BASE_WEIGHTS}"

## 4 · Stage 0b — stage the DeepPCB dataset

In [ ]:
DEEP = DATA / "deeppcb"
if not DEEP.exists():
    print("linking DeepPCB into working dir ...")
    link_tree(SRC_ROOT, DEEP)

# resolve split dir names (Roboflow uses train/valid/test)
def find_split(root, *cands):
    for c in cands:
        if (root / c / "images").exists():
            return root / c
    raise FileNotFoundError(f"no split among {cands} under {root}")

d_train = find_split(DEEP, "train")
d_val   = find_split(DEEP, "valid", "val")
d_test  = find_split(DEEP, "test")

DEEP_YAML = write_yaml(DATA / "deeppcb.yaml",
                       d_train / "images", d_val / "images", d_test / "images",
                       SOURCE_NAMES)
print(DEEP_YAML.read_text())
for d in (d_train, d_val, d_test):
    print(d.name, len(list((d / "images").iterdir())), "images")

## 5 · Stage 0c — train the source model on DeepPCBSkipped automatically if Stage 0a found a 6-class checkpoint.At `SRC_EPOCHS = 10` this takes roughly 10–15 min on a T4.> **Read this before interpreting any result.** A 10-epoch source model is a *pilot*, not a> converged model. Everything downstream inherits its weakness: the zero-shot number will be> pessimistic, and the apparent gain from fine-tuning will be inflated, because part of what the> fine-tuning recovers is simply source training the model never received. Treat this run as a> check that the plumbing works end to end. For numbers you would put in a paper, raise> `SRC_EPOCHS` to 100+ and confirm the DeepPCB test mAP in Stage 0d is in a sane range> (published DeepPCB detectors reach roughly 0.90+ mAP@50) before trusting the transfer results.

In [ ]:
from ultralytics import YOLO
import time

SRC_RUN = "source_deeppcb"
best_src = RUNS / "detect" / SRC_RUN / "weights" / "best.pt"

if source_ready is not None:
    SOURCE_WEIGHTS = source_ready
    print("using existing source weights:", SOURCE_WEIGHTS)
elif best_src.exists():
    SOURCE_WEIGHTS = best_src
    print("using weights from a previous run in this session:", SOURCE_WEIGHTS)
else:
    t0 = time.time()
    m = YOLO(str(BASE_WEIGHTS))
    m.train(data=str(DEEP_YAML), epochs=SRC_EPOCHS, imgsz=SRC_IMGSZ, batch=SRC_BATCH,
            project=str(RUNS / "detect"), name=SRC_RUN, exist_ok=True,
            seed=0, deterministic=True, patience=50, val=True, plots=True, verbose=True)
    SOURCE_WEIGHTS = best_src
    print(f"trained in {(time.time()-t0)/60:.1f} min ->", SOURCE_WEIGHTS)

assert Path(SOURCE_WEIGHTS).exists()

## 6 · Stage 0d — source-domain reference score (DeepPCB test split)

In [ ]:
src_model = YOLO(str(SOURCE_WEIGHTS))
print("source model classes:", src_model.names)

m_src = src_model.val(data=str(DEEP_YAML), split="test", imgsz=SRC_IMGSZ,
                      project=str(RUNS / "val"), name="source_on_deeppcb", exist_ok=True,
                      verbose=False)
tab_src = per_class_table(m_src, SOURCE_NAMES)
show_table(tab_src, "DeepPCB test (source domain):")
src_all    = subset_map(tab_src, list(range(6)))
src_shared = subset_map(tab_src, SHARED_IDX)
print(f"\n  all-6   mAP50={src_all['mAP50']:.3f}  mAP50-95={src_all['mAP50_95']:.3f}")
print(f"  shared-5 mAP50={src_shared['mAP50']:.3f}  mAP50-95={src_shared['mAP50_95']:.3f}")

## 7 · Stage 1 — build the target datasetCOCO → YOLO with the slot mapping, then a stratified 70/15/15 split.The Roboflow dummy category `detecting-pcb-defects` (id 0, zero annotations) is dropped.Splits are written as **image-list `.txt` files** over one flat `images/` + `labels/` pair, so the10% / 20% subsets cost no disk and every setting evaluates on the identical test split.

In [ ]:
coco = json.loads(TGT_COCO.read_text())
cats = {c["id"]: c["name"] for c in coco["categories"]}
print("COCO categories:", cats)

# map coco category id -> our target index (drop anything not in COCO_NAME_TO_IDX)
cid2idx, dropped = {}, []
for cid, name in cats.items():
    key = name.strip().lower()
    if key in COCO_NAME_TO_IDX:
        cid2idx[cid] = COCO_NAME_TO_IDX[key]
    else:
        dropped.append((cid, name))
print("dropped categories:", dropped)
assert len(cid2idx) == 6, f"expected 6 mapped categories, got {cid2idx}"

imgs = {im["id"]: im for im in coco["images"]}
anns_by_img = defaultdict(list)
for a in coco["annotations"]:
    if a.get("iscrowd", 0):
        continue
    if a["category_id"] in cid2idx:
        anns_by_img[a["image_id"]].append(a)

PCB   = DATA / "pcbdefect"
PIMG  = PCB / "images"
PLAB  = PCB / "labels"
PSPL  = PCB / "splits"
for d in (PIMG, PLAB, PSPL):
    d.mkdir(parents=True, exist_ok=True)

n_boxes, n_clipped, img_classes = 0, 0, {}
for iid, im in imgs.items():
    fn = im["file_name"]
    src_img = TGT_IMGDIR / fn
    if not src_img.exists():
        print("MISSING image, skipped:", fn); continue
    dst_img = PIMG / fn
    if not dst_img.exists():
        try: dst_img.symlink_to(src_img)
        except OSError: shutil.copy2(src_img, dst_img)

    W, H = im["width"], im["height"]
    lines, cc = [], Counter()
    for a in anns_by_img.get(iid, []):
        x, y, w, h = a["bbox"]
        x2, y2 = x + w, y + h
        cx0, cy0, cx1, cy1 = max(0, x), max(0, y), min(W, x2), min(H, y2)
        if cx1 <= cx0 or cy1 <= cy0:
            continue
        if (cx0, cy0, cx1, cy1) != (x, y, x2, y2):
            n_clipped += 1
        cx, cy = (cx0 + cx1) / 2 / W, (cy0 + cy1) / 2 / H
        bw, bh = (cx1 - cx0) / W, (cy1 - cy0) / H
        k = cid2idx[a["category_id"]]
        lines.append(f"{k} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        cc[k] += 1
        n_boxes += 1
    (PLAB / (Path(fn).stem + ".txt")).write_text("\n".join(lines) + ("\n" if lines else ""))
    img_classes[fn] = cc

print(f"\nconverted {len(img_classes)} images, {n_boxes} boxes, {n_clipped} clipped to bounds")
tot = Counter()
for c in img_classes.values(): tot.update(c)
for i, n in enumerate(TARGET_NAMES):
    print(f"  {i} {n:<18} {tot[i]:>5}")

### 7.1 Stratified split

In [ ]:
def split_score(assign, img_classes, fracs):
    # chi-square-like deviation of per-class counts from the expected proportions
    tot = Counter()
    for c in img_classes.values(): tot.update(c)
    per = {k: Counter() for k in fracs}
    for fn, k in assign.items(): per[k].update(img_classes[fn])
    s = 0.0
    for k, f in fracs.items():
        for c, n in tot.items():
            exp = n * f
            s += (per[k][c] - exp) ** 2 / max(exp, 1e-6)
    return s

files = sorted(img_classes)
n = len(files)
n_train = int(round(SPLIT_FRACS["train"] * n))
n_val   = int(round(SPLIT_FRACS["val"] * n))

best_assign, best_s = None, float("inf")
for trial in range(300):
    rng = random.Random(SPLIT_SEED + trial)
    f = files[:]; rng.shuffle(f)
    a = {}
    for i, fn in enumerate(f):
        a[fn] = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
    s = split_score(a, img_classes, SPLIT_FRACS)
    if s < best_s:
        best_assign, best_s = a, s
print(f"best split deviation score: {best_s:.3f} over 300 trials")

splits = defaultdict(list)
for fn, k in best_assign.items(): splits[k].append(fn)
for k in splits: splits[k].sort()

def write_list(name, filenames):
    p = PSPL / f"{name}.txt"
    p.write_text("\n".join(str(PIMG / f) for f in filenames) + "\n")
    return p

split_files = {k: write_list(k, v) for k, v in splits.items()}

print(f"\n{'split':<8}{'imgs':>6}  " + "".join(f"{n[:9]:>11}" for n in TARGET_NAMES))
for k in ("train", "val", "test"):
    cc = Counter()
    for fn in splits[k]: cc.update(img_classes[fn])
    print(f"{k:<8}{len(splits[k]):>6}  " + "".join(f"{cc[i]:>11}" for i in range(6)))

TGT_YAML = write_yaml(DATA / "pcbdefect.yaml",
                      split_files["train"], split_files["val"], split_files["test"],
                      TARGET_NAMES)
print("\n" + TGT_YAML.read_text())

### 7.2 Low-data subsetsAt 10% the train split is only ~16 images, so plain random sampling can leave a class with one orzero instances and produce a meaningless per-class AP. This picks images greedily to guarantee afloor of `MIN_PER_CLASS` instances per class wherever the data allows.

In [ ]:
def sample_subset(pool, img_classes, frac, seed, min_per_class=MIN_PER_CLASS):
    k = max(1, int(round(frac * len(pool))))
    rng = random.Random(seed)
    remaining = list(pool); rng.shuffle(remaining)
    chosen, cur = [], Counter()
    while len(chosen) < k and remaining:
        def score(fn):
            s = 0.0
            for c, cnt in img_classes[fn].items():
                deficit = max(0, min_per_class - cur[c])
                s += min(cnt, deficit) * 10.0 + cnt * 0.01
            return s
        remaining.sort(key=score, reverse=True)
        pick = remaining.pop(0)
        chosen.append(pick); cur.update(img_classes[pick])
    return sorted(chosen), cur

SUBSET_YAMLS, SUBSET_LISTS = {}, {}
print(f"{'setting':<16}{'imgs':>6}  " + "".join(f"{n[:9]:>11}" for n in TARGET_NAMES))
for frac in FRACTIONS:
    for seed in SEEDS:
        tag = f"f{int(frac*100):03d}_s{seed}"
        if frac >= 1.0:
            sel, cc = splits["train"], Counter()
            for fn in sel: cc.update(img_classes[fn])
        else:
            sel, cc = sample_subset(splits["train"], img_classes, frac, seed)
        lst = write_list(f"train_{tag}", sel)
        SUBSET_LISTS[(frac, seed)] = lst
        SUBSET_YAMLS[(frac, seed)] = write_yaml(
            DATA / f"pcbdefect_{tag}.yaml", lst, split_files["val"], split_files["test"], TARGET_NAMES)
        print(f"{tag:<16}{len(sel):>6}  " + "".join(f"{cc[i]:>11}" for i in range(6)))
        if frac >= 1.0:
            break   # full split is identical across seeds; data identical, only training seed varies

### 7.3 Sanity check — draw ground-truth boxes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#46f0f0"]

def draw_gt(fn, ax):
    img = Image.open(PIMG / fn); W, H = img.size
    ax.imshow(img); ax.set_title(fn, fontsize=8); ax.axis("off")
    for ln in (PLAB / (Path(fn).stem + ".txt")).read_text().split("\n"):
        if not ln.strip(): continue
        c, cx, cy, bw, bh = ln.split()
        c = int(c); cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
        ax.add_patch(mpatches.Rectangle((cx-bw/2, cy-bh/2), bw, bh, fill=False,
                                        edgecolor=COLORS[c], linewidth=1.6))

sel = splits["test"][:4]
fig, axes = plt.subplots(1, len(sel), figsize=(5*len(sel), 5))
for ax, fn in zip(np.atleast_1d(axes), sel): draw_gt(fn, ax)
handles = [mpatches.Patch(color=COLORS[i], label=TARGET_NAMES[i]) for i in range(6)]
fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False)
plt.tight_layout(); plt.savefig(OUT / "figs" / "gt_sanity.png", dpi=120, bbox_inches="tight"); plt.show()

## 8 · Stage 2 — zero-shot transferThe source model is applied to the target test split with **no training**.Evaluated at two input sizes: the target images are ~2000–3400 px wide while the source model wastrained at 640, so a large gain at 1280 would indicate the gap is substantially about **scale**rather than appearance.`mAP50 (shared-5)` is the headline. Class 3 is excluded — see the note at the top.

In [ ]:
zs_results = {}
for imgsz in ZS_IMGSZ:
    m = src_model.val(data=str(TGT_YAML), split="test", imgsz=imgsz,
                      project=str(RUNS / "val"), name=f"zeroshot_{imgsz}", exist_ok=True,
                      verbose=False)
    # NOTE: names come from the TARGET yaml, but index 3 predictions mean pin-hole.
    tab = per_class_table(m, TARGET_NAMES)
    zs_results[imgsz] = tab
    show_table(tab, f"\nZero-shot @ imgsz={imgsz} (target test)")
    sh = subset_map(tab, SHARED_IDX)
    al = subset_map(tab, list(range(6)))
    print(f"  --> shared-5 mAP50={sh['mAP50']:.4f}  mAP50-95={sh['mAP50_95']:.4f}   [HEADLINE]")
    print(f"      all-6    mAP50={al['mAP50']:.4f}  (slot class 3 is NOT comparable)")

ZS_BEST_IMGSZ = max(ZS_IMGSZ, key=lambda s: subset_map(zs_results[s], SHARED_IDX)["mAP50"])
print("\nbest zero-shot input size:", ZS_BEST_IMGSZ)

### 8.1 Diagnostic — does the `pin-hole` neuron respond to `missing_pad`?Both defects are "copper that should be present but isn't", so cross-firing is plausible.This matches every class-3 prediction from the source model against `missing_pad` ground truth atIoU ≥ 0.3 and reports the hit rate, plus what the source model predicts over missing-pad regionsgenerally.

In [ ]:
def iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw*ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

def load_gt(fn):
    img = Image.open(PIMG / fn); W, H = img.size
    out = []
    for ln in (PLAB / (Path(fn).stem + ".txt")).read_text().split("\n"):
        if not ln.strip(): continue
        c, cx, cy, bw, bh = ln.split()
        cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
        out.append((int(c), [cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2]))
    return out

IOU_T, CONF_T = 0.30, 0.15
n_mp, n_mp_hit = 0, 0
pred_over_mp = Counter()
cls3_total, cls3_on_mp = 0, 0

for fn in splits["test"]:
    r = src_model.predict(str(PIMG / fn), imgsz=ZS_BEST_IMGSZ, conf=CONF_T, verbose=False)[0]
    boxes = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.zeros((0, 4))
    clss  = r.boxes.cls.cpu().numpy().astype(int) if r.boxes is not None else np.zeros((0,), int)
    gts = load_gt(fn)
    cls3_total += int((clss == SLOT_IDX).sum())
    for gc, gb in gts:
        if gc != SLOT_IDX:
            continue
        n_mp += 1
        hit = False
        for pb, pc in zip(boxes, clss):
            if iou_xyxy(gb, pb) >= IOU_T:
                pred_over_mp[int(pc)] += 1
                if pc == SLOT_IDX:
                    hit = True
        if hit:
            n_mp_hit += 1

print(f"missing_pad GT boxes in test split : {n_mp}")
print(f"  matched by a 'pin-hole' (idx 3) prediction at IoU>={IOU_T}: {n_mp_hit} "
      f"({100*n_mp_hit/max(n_mp,1):.1f}%)")
print(f"  total idx-3 predictions on the split: {cls3_total}")
print("\nsource-model predictions overlapping missing_pad regions, by SOURCE class name:")
for c, k in pred_over_mp.most_common():
    print(f"  {SOURCE_NAMES[c]:<18}{k:>5}")

## 9 · Stage 3 — fine-tuning at 10% / 20% / 100%Every run starts from the same source weights and is evaluated on the **identical** test split.`nc` is 6 on both sides, so the full classification head is reused.

In [ ]:
import pandas as pd, time

rows = []
ft_weights = {}

for frac in FRACTIONS:
    for seed in SEEDS:
        key = (frac, seed) if (frac, seed) in SUBSET_YAMLS else (frac, SEEDS[0])
        yml = SUBSET_YAMLS[key]
        tag = f"ft_f{int(frac*100):03d}_s{seed}"
        wpath = RUNS / "detect" / tag / "weights" / "best.pt"

        # --- iteration-matched schedule -------------------------------------
        # A fixed epoch count is unfair to the small fractions: at 10% the train
        # split is ~16 images, so one epoch is 2 optimizer steps at batch=8, and
        # 50 epochs is only ~100 steps -- less than the LR warm-up needs. Scale
        # epochs so every fraction gets a comparable number of optimizer steps,
        # shorten the warm-up, and scale patience with the schedule so early
        # stopping cannot fire before the model has had a chance to move.
        n_imgs = len(Path(str(SUBSET_LISTS[key])).read_text().split())
        bs = max(2, min(FT_BATCH, n_imgs))
        ep = int(round(FT_EPOCHS * (max(FRACTIONS) / frac)))
        pat = max(FT_PATIENCE, ep // 3)

        if not wpath.exists():
            t0 = time.time()
            m = YOLO(str(SOURCE_WEIGHTS))
            m.train(data=str(yml), epochs=ep, imgsz=FT_IMGSZ, batch=bs,
                    project=str(RUNS / "detect"), name=tag, exist_ok=True,
                    seed=seed, deterministic=True, patience=pat,
                    warmup_epochs=1.0, val=True, plots=False, verbose=False)
            print(f"[{tag}] {n_imgs} imgs, batch={bs}, epochs={ep}, patience={pat} "
                  f"-> {(time.time()-t0)/60:.1f} min")
        ft_weights[(frac, seed)] = wpath

        mm = YOLO(str(wpath))
        met = mm.val(data=str(TGT_YAML), split="test", imgsz=FT_IMGSZ,
                     project=str(RUNS / "val"), name=tag, exist_ok=True, verbose=False)
        tab = per_class_table(met, TARGET_NAMES)
        sh, al = subset_map(tab, SHARED_IDX), subset_map(tab, list(range(6)))
        rows.append({"setting": f"finetune_{int(frac*100)}%", "frac": frac, "seed": seed,
                     "mAP50_all6": al["mAP50"], "mAP5095_all6": al["mAP50_95"],
                     "mAP50_shared5": sh["mAP50"], "mAP5095_shared5": sh["mAP50_95"],
                     "P": al["P"], "R": al["R"],
                     **{f"AP50_{TARGET_NAMES[i]}": tab.get(i, {}).get("mAP50", float('nan'))
                        for i in range(6)}})
        print(f"[{tag}] mAP50 all6={al['mAP50']:.4f}  shared5={sh['mAP50']:.4f}")

df_ft = pd.DataFrame(rows)
df_ft.to_csv(OUT / "finetune_runs.csv", index=False)
df_ft

## 10 · Stage 4 — results

In [ ]:
# zero-shot rows in the same schema
zs_rows = []
for imgsz, tab in zs_results.items():
    sh, al = subset_map(tab, SHARED_IDX), subset_map(tab, list(range(6)))
    zs_rows.append({"setting": f"zero-shot@{imgsz}", "frac": 0.0, "seed": -1,
                    "mAP50_all6": al["mAP50"], "mAP5095_all6": al["mAP50_95"],
                    "mAP50_shared5": sh["mAP50"], "mAP5095_shared5": sh["mAP50_95"],
                    "P": al["P"], "R": al["R"],
                    **{f"AP50_{TARGET_NAMES[i]}": tab.get(i, {}).get("mAP50", float('nan'))
                       for i in range(6)}})
df_all = pd.concat([pd.DataFrame(zs_rows), df_ft], ignore_index=True)
df_all.to_csv(OUT / "all_runs.csv", index=False)

summary = (df_all.groupby("setting")
           .agg(n=("seed", "count"),
                mAP50_shared5_mean=("mAP50_shared5", "mean"),
                mAP50_shared5_std=("mAP50_shared5", "std"),
                mAP50_all6_mean=("mAP50_all6", "mean"),
                mAP50_all6_std=("mAP50_all6", "std"),
                mAP5095_all6_mean=("mAP5095_all6", "mean"))
           .reset_index())
summary.to_csv(OUT / "summary.csv", index=False)
print("Source reference (DeepPCB test): "
      f"mAP50 all6={src_all['mAP50']:.3f}  shared5={src_shared['mAP50']:.3f}\n")
summary

In [ ]:
# adaptation curve
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

zs_sh = subset_map(zs_results[ZS_BEST_IMGSZ], SHARED_IDX)["mAP50"]
g = df_ft.groupby("frac")["mAP50_shared5"].agg(["mean", "std"]).reset_index()
x = [0] + [f*100 for f in g["frac"]]
y = [zs_sh] + list(g["mean"])
e = [0] + list(g["std"].fillna(0))
ax[0].errorbar(x, y, yerr=e, marker="o", capsize=4, color="#4363d8")
ax[0].axhline(src_shared["mAP50"], ls="--", c="grey",
              label=f"source domain (DeepPCB) = {src_shared['mAP50']:.3f}")
ax[0].set_xlabel("% of target train split used"); ax[0].set_ylabel("mAP@50 (shared-5)")
ax[0].set_title("Adaptation curve"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

# per-class, zero-shot vs best fine-tune
best_frac = g.loc[g["mean"].idxmax(), "frac"]
per_ft = df_ft[df_ft.frac == best_frac][[f"AP50_{n}" for n in TARGET_NAMES]].mean()
per_zs = [zs_results[ZS_BEST_IMGSZ].get(i, {}).get("mAP50", 0.0) for i in range(6)]
w, xs = 0.38, np.arange(6)
ax[1].bar(xs - w/2, per_zs, w, label=f"zero-shot @{ZS_BEST_IMGSZ}", color="#f58231")
ax[1].bar(xs + w/2, per_ft.values, w, label=f"fine-tune {int(best_frac*100)}%", color="#3cb44b")
ax[1].set_xticks(xs); ax[1].set_xticklabels(TARGET_NAMES, rotation=30, ha="right", fontsize=8)
ax[1].set_ylabel("AP@50"); ax[1].set_title("Per-class"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis="y")
ax[1].annotate("slot class\n(not comparable)", xy=(3, 0.02), fontsize=7, ha="center", color="crimson")

plt.tight_layout(); plt.savefig(OUT / "figs" / "adaptation_curve.png", dpi=140, bbox_inches="tight")
plt.show()

## 11 · Stage 5 — XAI: EigenCAM, Grad-CAM, Grad-CAM++ and HiResCAM### What was wrong with the first attemptTwo independent faults, both visible in the earlier figures:1. **Grad-CAM returned an all-zero map.** Vanilla Grad-CAM weights each channel by the *mean*   gradient, `w_k = mean(dS/dA_k)`, then applies `ReLU(Σ w_k A_k)`. Those means are signed, and on   this detector they came out predominantly negative, so the ReLU erased the entire map. This is a   known Grad-CAM failure mode, not a bug in the plumbing — and exactly why **Grad-CAM++** exists:   it weights by `Σ α · ReLU(dS/dA)`, which is non-negative by construction and cannot collapse   this way.2. **The map was computed at the wrong scale.** Layer 22 is the **P5** branch: a 20×20 grid over a   640-px input, so one cell covers 32×32 input pixels — which on a 2000-px board is ~100 real   pixels. PCB defects are a few pixels across. No amount of upsampling recovers detail that the   20×20 grid never had, which is why the old maps were smooth blobs unrelated to the small boxes.### What this section does instead* CAMs are computed at **P3 (80×80, layer 16)** and **P4 (40×40, layer 19)** and fused — P3 is the  small-object branch and is where defect evidence actually lives. P5 is available but off by  default.* Four methods share one forward/backward pass: **EigenCAM**, **Grad-CAM**, **Grad-CAM++**,  **HiResCAM** (`Σ A ⊙ G`, which keeps per-pixel gradient sign information instead of averaging it  away — often the sharpest of the four on small objects).* The differentiated scalar is the sum of the **top-k detection confidences** taken from the head's  `one2one` scores after sigmoid, so it is bounded in [0,1] and always positive.* A degenerate map (all-zero after ReLU) is detected, reported, and falls back to `|Σ w_k A_k|`  rather than silently rendering a flat image.* Display uses **percentile clipping** (2–99%) so one hot pixel cannot flatten everything else, and  the overlay alpha scales with intensity so the board stays readable under cold regions.### Quantifying "does it hit the right place"Pictures alone are not evidence. Section 11.6 computes, for every method and model:* **Pointing-game accuracy** — is the CAM's global maximum inside any ground-truth box?* **Energy ratio** — fraction of total CAM mass falling inside ground-truth boxes.* **Area baseline** — fraction of the image the boxes cover, i.e. what a *uniform random* map  would score. An energy ratio at or below this number means the explanation carries no  localization signal, however striking the picture looks.

In [ ]:
import torch.nn as nn, cv2

_seq = YOLO(str(SOURCE_WEIGHTS)).model.model
print(f"{len(_seq)} modules; last 12:")
for i in range(len(_seq)-12, len(_seq)):
    print(f"  [{i}] {_seq[i].__class__.__name__}")

# P3 = fine (80x80, small objects), P4 = medium (40x40), P5 = coarse (20x20)
CAM_LAYERS = [16, 19]          # add 22 for P5 as well
print("\nCAM layers:", [(i, _seq[i].__class__.__name__) for i in CAM_LAYERS])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METHODS = ["EigenCAM", "Grad-CAM", "Grad-CAM++", "HiResCAM"]

### 11.1 CAM engine

In [ ]:
class CAM:
    # One forward (+ one backward) serves every gradient-based method.
    def __init__(self, weights, layers=CAM_LAYERS, imgsz=640, device=DEVICE, topk=10):
        self.net = YOLO(str(weights)).model.float().to(device).eval()
        # Ultralytics loads inference models with requires_grad=False everywhere
        # (the "0 gradients" line in the model summary) - without this the
        # forward pass builds no graph and backward() fails.
        for p in self.net.parameters():
            p.requires_grad_(True)
        self.nc = int(getattr(self.net, "nc", 6))
        self.imgsz, self.device, self.layers, self.topk = imgsz, device, list(layers), topk
        self.acts = {}
        for li in self.layers:
            self.net.model[li].register_forward_hook(self._mk_hook(li))
        self.warned = set()

    def _mk_hook(self, li):
        def hook(module, inp, out):
            a = out[0] if isinstance(out, (list, tuple)) else out
            self.acts[li] = a
            if a.requires_grad:
                a.retain_grad()
        return hook

    def _prep(self, img_path):
        bgr = cv2.imread(str(img_path))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        x = cv2.resize(rgb, (self.imgsz, self.imgsz), interpolation=cv2.INTER_LINEAR)
        t = torch.from_numpy(x).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(self.device)
        return t, rgb

    def _scores(self, out):
        # YOLO26 returns (preds, {one2many:{boxes,scores,feats}, one2one:{...}})
        d = out if isinstance(out, dict) else None
        if d is None and isinstance(out, (list, tuple)):
            d = next((e for e in out if isinstance(e, dict)), None)
        if d is not None:
            head = d.get("one2one", d.get("one2many"))
            if isinstance(head, dict) and "scores" in head:
                s = head["scores"]                                  # (B, nc, A)
                if s.min() < 0 or s.max() > 1:
                    s = s.sigmoid()                                 # logits -> probabilities
                return s.amax(dim=1).reshape(s.shape[0], -1)
        t = out
        while isinstance(t, (list, tuple)) and len(t):
            t = t[0]
        if torch.is_tensor(t) and t.ndim == 3:
            if t.shape[1] == 4 + self.nc:
                return t[:, 4:, :].amax(dim=1).reshape(t.shape[0], -1)
            if t.shape[2] == 4 + self.nc:
                return t[..., 4:].amax(dim=-1).reshape(t.shape[0], -1)
            if t.shape[2] == 6:
                return t[..., 4].reshape(t.shape[0], -1)
        raise RuntimeError("could not locate detection scores in model output")

    def _run(self, img_path, need_grad):
        t, rgb = self._prep(img_path)
        self.acts.clear()
        if need_grad:
            self.net.zero_grad(set_to_none=True)
            t.requires_grad_(True)
            with torch.enable_grad():
                out = self.net(t)
                s = self._scores(out)[0]
                k = min(self.topk, s.numel())
                s.topk(k).values.sum().backward()
        else:
            with torch.no_grad():
                self.net(t)
        return rgb

    # ---- per-layer maps ----------------------------------------------------
    def _layer_map(self, li, method):
        a = self.acts[li]
        A = a.detach()[0].float()                       # (C, H, W)
        if method == "EigenCAM":
            C, H, W = A.shape
            flat = A.reshape(C, -1)
            flat = flat - flat.mean(dim=1, keepdim=True)
            _, _, Vh = torch.linalg.svd(flat, full_matrices=False)
            cam = Vh[0].reshape(H, W)
            if cam.mean() < 0:
                cam = -cam
            return torch.relu(cam)

        G = a.grad
        if G is None:
            raise RuntimeError(f"no gradient at layer {li} - was _run(need_grad=True) used?")
        G = G.detach()[0].float()

        if method == "Grad-CAM":
            w = G.mean(dim=(1, 2), keepdim=True)
            raw = (w * A).sum(0)
        elif method == "Grad-CAM++":
            G2, G3 = G * G, G * G * G
            denom = 2.0 * G2 + (A * G3).sum(dim=(1, 2), keepdim=True)
            alpha = G2 / torch.where(denom.abs() < 1e-12, torch.full_like(denom, 1e-12), denom)
            w = (alpha * torch.relu(G)).sum(dim=(1, 2), keepdim=True)
            raw = (w * A).sum(0)
        elif method == "HiResCAM":
            raw = (A * G).sum(0)
        else:
            raise ValueError(method)

        cam = torch.relu(raw)
        if float(cam.max()) <= 0:                       # degenerate: ReLU erased everything
            key = (method, li)
            if key not in self.warned:
                print(f"  [note] {method} @layer {li}: all-negative map, using |.| fallback")
                self.warned.add(key)
            cam = raw.abs()
        return cam

    # ---- public ------------------------------------------------------------
    def cam(self, img_path, method):
        rgb = self._run(img_path, need_grad=(method != "EigenCAM"))
        maps = []
        for li in self.layers:
            c = self._layer_map(li, method).cpu().numpy()
            maps.append(_unit(c))
        S = max(m.shape[0] for m in maps)
        maps = [cv2.resize(m, (S, S), interpolation=cv2.INTER_CUBIC) for m in maps]
        return _unit(np.mean(maps, axis=0)), rgb

    def all_cams(self, img_path, methods=METHODS):
        return {m: self.cam(img_path, m)[0] for m in methods}


def _unit(c, lo=2.0, hi=99.0):
    # percentile clipping: one hot cell must not flatten the rest of the map
    c = np.nan_to_num(np.asarray(c, dtype=np.float32))
    a, b = np.percentile(c, lo), np.percentile(c, hi)
    if b - a < 1e-8:
        a, b = float(c.min()), float(c.max())
    c = np.clip((c - a) / (b - a + 1e-8), 0, 1)
    return c


def overlay(rgb, cam, amax=0.75, amin=0.10, blur=0):
    # alpha scales with intensity so cold regions stay legible
    h, w = rgb.shape[:2]
    cm = cv2.resize(cam, (w, h), interpolation=cv2.INTER_CUBIC)
    if blur:
        cm = cv2.GaussianBlur(cm, (0, 0), blur)
    cm = np.clip(cm, 0, 1)
    hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*cm), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    al = (amin + (amax - amin) * cm)[..., None]
    return np.uint8(al * hm + (1 - al) * rgb)


def gt_boxes(img_path, label_path):
    im = Image.open(img_path); W, H = im.size
    out = []
    p = Path(label_path)
    if not p.exists():
        return out
    for ln in p.read_text().split("\n"):
        if not ln.strip():
            continue
        c, cx, cy, bw, bh = ln.split()
        cx, cy, bw, bh = float(cx)*W, float(cy)*H, float(bw)*W, float(bh)*H
        out.append((int(c), [cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2]))
    return out

print("CAM engine ready")

#### Self-testVerifies every method produces a non-degenerate map at every layer, and prints the map resolutionso you can confirm you are on P3/P4 and not the coarse P5 grid.

In [ ]:
_probe = CAM(SOURCE_WEIGHTS, imgsz=SRC_IMGSZ)
_img = next(iter(sorted((d_test / "images").iterdir())))
for _m in METHODS:
    try:
        c, _ = _probe.cam(_img, _m)
        uniq = len(np.unique(np.round(c, 3)))
        flag = "OK" if (c.max() > 0 and uniq > 10) else "DEGENERATE"
        print(f"  {_m:<12} map{c.shape}  range[{c.min():.2f},{c.max():.2f}]  "
              f"distinct={uniq:<5} {flag}")
    except Exception as e:
        print(f"  {_m:<12} FAILED: {type(e).__name__}: {e}")
for li in _probe.layers:
    print(f"  layer {li}: activation {tuple(_probe.acts[li].shape)}")
del _probe

### 11.2 Figure builder

In [ ]:
def cam_figure(model_tags, samples, names, outfile, title,
               methods=METHODS, figscale=3.4, draw_gt_on_cam=True):
    # model_tags: list of (CAM object, short label)
    cols = 1 + len(methods) * len(model_tags)
    rows = len(samples)
    fig, axes = plt.subplots(rows, cols, figsize=(figscale*cols, figscale*rows*1.05), squeeze=False)
    for r, (ip, lp) in enumerate(samples):
        gt = gt_boxes(ip, lp)
        rgb = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)

        def boxes_on(ax, lw=1.6):
            for c, b in gt:
                ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                             fill=False, edgecolor=COLORS[c % 6], lw=lw))

        axes[r][0].imshow(rgb); boxes_on(axes[r][0])
        if r == 0:
            axes[r][0].set_title("ground truth", fontsize=11)
        axes[r][0].set_ylabel(Path(ip).stem, fontsize=8)

        col = 1
        for cobj, tag in model_tags:
            cams = cobj.all_cams(ip, methods)
            for m in methods:
                axes[r][col].imshow(overlay(rgb, cams[m]))
                if draw_gt_on_cam:
                    boxes_on(axes[r][col], lw=1.1)
                if r == 0:
                    axes[r][col].set_title(f"{m}\n{tag}", fontsize=10)
                col += 1
        for ax in axes[r]:
            ax.set_xticks([]); ax.set_yticks([])

    handles = [mpatches.Patch(color=COLORS[i], label=names[i]) for i in range(len(names))]
    fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False, fontsize=9)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(rect=[0, 0.035, 1, 0.965])
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.show()
    print("saved", outfile)

N_SHOW = 3
cam_source = CAM(SOURCE_WEIGHTS, imgsz=SRC_IMGSZ)

deep_samples = [(p, d_test / "labels" / (p.stem + ".txt"))
                for p in sorted((d_test / "images").iterdir())[:N_SHOW]]
tgt_samples  = [(PIMG / fn, PLAB / (Path(fn).stem + ".txt")) for fn in splits["test"][:N_SHOW]]
print("figure builder ready")

### 11.3 Figure 1 — pretrained model on its own domain (DeepPCB)The reference panel. At 0.98 mAP@50 the model is working, so this is what *correct* attentionlooks like — and the benchmark the target-domain figures are judged against.

In [ ]:
cam_figure([(cam_source, "pretrained")], deep_samples, SOURCE_NAMES,
           OUT / "figs" / "fig_xai_source_on_deeppcb.png",
           "Pretrained YOLO26n on the source domain (DeepPCB test)")

### 11.4 Figure 2 — same model, target domain (zero-shot)Shared-5 mAP@50 is ~0.015 here, so the model is effectively blind. The question these maps answeris *how* it is blind: attention spread over board texture, locked onto a few high-contraststructures, or concentrated confidently in the wrong places.

In [ ]:
cam_figure([(cam_source, "pretrained / zero-shot")], tgt_samples, TARGET_NAMES,
           OUT / "figs" / "fig_xai_source_on_pcbdefect.png",
           "Pretrained YOLO26n on the target domain (PCB-Defect v1 test, zero-shot)")

### 11.5 Figure 3 — before vs. after adaptationRestricted to EigenCAM and Grad-CAM++ so the panel stays readable; change `methods=` to show allfour.

In [ ]:
best_seed = int(df_ft[df_ft.frac == best_frac].sort_values("mAP50_shared5").iloc[-1]["seed"])
BEST_FT_W = ft_weights[(best_frac, best_seed)]
print("source :", SOURCE_WEIGHTS)
print("adapted:", BEST_FT_W, f"(fine-tune {int(best_frac*100)}%, seed {best_seed})")

cam_ft = CAM(BEST_FT_W, imgsz=FT_IMGSZ)

ba_samples = [(PIMG / fn, PLAB / (Path(fn).stem + ".txt")) for fn in splits["test"][:4]]
cam_figure([(cam_source, "pretrained"), (cam_ft, f"fine-tuned {int(best_frac*100)}%")],
           ba_samples, TARGET_NAMES,
           OUT / "figs" / "fig_xai_before_after.png",
           "Attention before and after target-domain adaptation (PCB-Defect v1 test)",
           methods=["EigenCAM", "Grad-CAM++"], figscale=3.6)

### 11.6 Quantitative XAI — does the map land on the defects?**Pointing game**: is the CAM's global maximum inside a ground-truth box?**Energy ratio**: what fraction of the CAM's total mass lies inside ground-truth boxes?**Area baseline**: what fraction of the image those boxes cover — the score a uniform random mapwould get. Energy ratio ≤ area baseline means the map carries no localization information.Report the ratio of energy to area (`lift`) in the paper; it is scale-free and interpretable:lift = 1.0 is chance, lift > 1 means genuine concentration on defects.

In [ ]:
def cam_metrics(cobj, samples, methods=METHODS, label=""):
    rows = []
    for m in methods:
        hits = tot = 0
        energies, areas = [], []
        for ip, lp in samples:
            gt = gt_boxes(ip, lp)
            if not gt:
                continue
            cam, rgb = cobj.cam(ip, m)
            H, W = rgb.shape[:2]
            cm = cv2.resize(cam, (W, H), interpolation=cv2.INTER_CUBIC)
            cm = np.clip(cm, 0, None)
            y, x = np.unravel_index(int(cm.argmax()), cm.shape)
            hits += int(any(b[0] <= x <= b[2] and b[1] <= y <= b[3] for _, b in gt))
            tot += 1
            mask = np.zeros(cm.shape, bool)
            for _, b in gt:
                x0, y0 = max(0, int(b[0])), max(0, int(b[1]))
                x1, y1 = min(W, int(np.ceil(b[2]))), min(H, int(np.ceil(b[3])))
                mask[y0:y1, x0:x1] = True
            tot_e = cm.sum()
            energies.append(cm[mask].sum() / (tot_e + 1e-9))
            areas.append(mask.mean())
        e, a = float(np.mean(energies)), float(np.mean(areas))
        rows.append({"model": label, "method": m,
                     "pointing_acc": hits / max(tot, 1),
                     "energy_ratio": e, "area_baseline": a,
                     "lift": e / (a + 1e-9), "n_images": tot})
    return rows

metric_rows = []
metric_rows += cam_metrics(cam_source, deep_samples + [
    (p, d_test / "labels" / (p.stem + ".txt"))
    for p in sorted((d_test / "images").iterdir())[N_SHOW:12]], label="pretrained / DeepPCB")
metric_rows += cam_metrics(cam_source, [(PIMG / fn, PLAB / (Path(fn).stem + ".txt"))
                                        for fn in splits["test"]], label="pretrained / target")
metric_rows += cam_metrics(cam_ft, [(PIMG / fn, PLAB / (Path(fn).stem + ".txt"))
                                    for fn in splits["test"]],
                           label=f"fine-tuned {int(best_frac*100)}% / target")

df_xai = pd.DataFrame(metric_rows)
df_xai.to_csv(OUT / "xai_metrics.csv", index=False)
pd.set_option("display.width", 160)
print(df_xai.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# lift chart: >1 means the map concentrates on defects more than chance
fig, ax = plt.subplots(figsize=(9, 4.2))
models = df_xai["model"].unique()
w = 0.8 / len(models)
xs = np.arange(len(METHODS))
for i, mdl in enumerate(models):
    sub = df_xai[df_xai.model == mdl].set_index("method").reindex(METHODS)
    ax.bar(xs + i*w - 0.4 + w/2, sub["lift"].values, w, label=mdl)
ax.axhline(1.0, ls="--", c="crimson", lw=1.2, label="chance (uniform map)")
ax.set_xticks(xs); ax.set_xticklabels(METHODS)
ax.set_ylabel("energy / area  (lift)")
ax.set_title("Do the heat maps concentrate on annotated defects?")
ax.legend(fontsize=8); ax.grid(alpha=.3, axis="y")
plt.tight_layout()
plt.savefig(OUT / "figs" / "fig_xai_lift.png", dpi=200, bbox_inches="tight")
plt.show()

### 11.7 Save every heat map individually (for figure assembly)

In [ ]:
XD = OUT / "xai"
SAVE_METHODS = ["EigenCAM", "Grad-CAM++", "HiResCAM"]
for sub in ("deeppcb_source", "pcbdefect_source", "pcbdefect_finetuned"):
    (XD / sub).mkdir(parents=True, exist_ok=True)

def dump(cobj, samples, subdir, methods=SAVE_METHODS):
    for ip, _ in samples:
        cams = cobj.all_cams(ip, methods)
        rgb = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        for m, c in cams.items():
            fn = XD / subdir / f"{Path(ip).stem}_{m.replace('+','p').replace('-','').lower()}.jpg"
            cv2.imwrite(str(fn), cv2.cvtColor(overlay(rgb, c), cv2.COLOR_RGB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])

deep_all = [(p, d_test / "labels" / (p.stem + ".txt"))
            for p in sorted((d_test / "images").iterdir())[:12]]
tgt_all  = [(PIMG / fn, PLAB / (Path(fn).stem + ".txt")) for fn in splits["test"]]

dump(cam_source, deep_all, "deeppcb_source")
dump(cam_source, tgt_all,  "pcbdefect_source")
dump(cam_ft,     tgt_all,  "pcbdefect_finetuned")
print("saved", len(list(XD.rglob("*.jpg"))), "heat maps under", XD)

## 12 · Package the outputs

In [ ]:
import shutil
# copy the best weights next to the results so they survive the session
for name, w in [("source_deeppcb_best.pt", SOURCE_WEIGHTS), ("finetuned_best.pt", BEST_FT_W)]:
    try: shutil.copy2(w, OUT / name)
    except Exception as e: print("skip", name, e)

shutil.make_archive(str(WORK / "pcb_adaptation_results"), "zip", str(OUT))
print("wrote /kaggle/working/pcb_adaptation_results.zip")
for p in sorted(OUT.rglob("*")):
    if p.is_file(): print(" ", p.relative_to(OUT), f"{p.stat().st_size/1e3:.0f} KB")